In [ ]:
!pip -q install PyWavelets tqdm

In [ ]:
import os
import zipfile
import warnings

import numpy as np
import pandas as pd
import pywt

from tqdm.auto import tqdm

from scipy.fft import rfft, dct
from scipy.signal import stft, resample

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ===============================
# CHANGE ONLY THIS IF REQUIRED
# ===============================

ZIP_PATH = "/content/drive/MyDrive/EE656project/AirCompressor_Data.zip"

# Dataset will be extracted here
EXTRACT_PATH = "/content"

# Final extracted folder
DATASET_PATH = "/content/AirCompressor_Data"

# Output files
OUTPUT_CSV = "/content/drive/MyDrive/EE656project/features.csv"

CHECKPOINT_CSV = "/content/drive/MyDrive/EE656project/features_checkpoint.csv"

In [ ]:
# Extract only once

if not os.path.exists(DATASET_PATH):

    print("Extracting dataset...")

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:

        zip_ref.extractall(EXTRACT_PATH)

    print("Extraction completed!")

else:

    print("Dataset already extracted.")

Extracting dataset...
Extraction completed!


In [ ]:
folders = sorted(os.listdir(DATASET_PATH))

print("Folders Found:\n")

for folder in folders:
    print(folder)

Folders Found:

Bearing
Flywheel
Healthy
LIV
LOV
NRV
Piston
Riderbelt


In [ ]:
CLASS_NAMES = [

    "Bearing",

    "Flywheel",

    "Healthy",

    "LIV",

    "LOV",

    "NRV",

    "Piston",

    "Riderbelt"

]

CLASS_TO_LABEL = {

    cls: idx

    for idx, cls in enumerate(CLASS_NAMES)

}

In [ ]:
def load_dat(filepath):

    signal = np.loadtxt(

        filepath,

        delimiter=',',

        dtype=np.float64

    )

    signal = np.asarray(signal).flatten()

    return signal

In [ ]:
def scan_dataset(root):

    dataset = []

    for cls in CLASS_NAMES:

        folder = os.path.join(root, cls)

        files = sorted(

            f for f in os.listdir(folder)

            if f.endswith(".dat")

        )

        for file in files:

            dataset.append({

                "filename": file,

                "filepath": os.path.join(folder, file),

                "class": cls,

                "label": CLASS_TO_LABEL[cls]

            })

    return dataset

In [ ]:
dataset = scan_dataset(DATASET_PATH)

print(f"Total Files : {len(dataset)}")

print()

print(dataset[0])

Total Files : 1800

{'filename': 'preprocess_Reading1.dat', 'filepath': '/content/AirCompressor_Data/Bearing/preprocess_Reading1.dat', 'class': 'Bearing', 'label': 0}


In [ ]:
signal = load_dat(dataset[0]["filepath"])

print("Filename :", dataset[0]["filename"])

print("Signal Length :", len(signal))

print("Datatype :", signal.dtype)

print()

print(signal[:10])

Filename : preprocess_Reading1.dat
Signal Length : 50000
Datatype : float64

[-0.072547 -0.025741  0.031091  0.13126   0.24657   0.34774   0.41756
  0.45302   0.4611    0.45327 ]


In [ ]:
def energy_features(values, n_bins):

    values = np.asarray(values, dtype=np.float64)

    energy = values**2

    total_energy = np.sum(energy)

    if total_energy == 0:

        return np.zeros(n_bins)

    bins = np.array_split(energy, n_bins)

    features = np.array(

        [

            np.sum(b)

            for b in bins

        ],

        dtype=np.float64

    )

    features /= total_energy

    return features

In [ ]:
print("="*60)

print(f"Total Recordings : {len(dataset)}")

print()

for cls in CLASS_NAMES:

    n = sum(item["class"] == cls for item in dataset)

    print(f"{cls:12s} : {n}")

print()

print("="*60)

Total Recordings : 1800

Bearing      : 225
Flywheel     : 225
Healthy      : 225
LIV          : 225
LOV          : 225
NRV          : 225
Piston       : 225
Riderbelt    : 225



In [ ]:
# ==========================================================
# FFT FEATURE EXTRACTION
# ==========================================================

def extract_fft_features(signal):
    """
    Extract 8 normalized FFT energy features.
    """

    # Positive-frequency FFT
    fft_coeffs = np.abs(rfft(signal))

    # 8 normalized energy bins
    return energy_features(fft_coeffs, n_bins=8)

In [ ]:
# ==========================================================
# DCT FEATURE EXTRACTION
# ==========================================================

def extract_dct_features(signal):
    """
    Extract 8 normalized DCT energy features.
    """

    coeffs = np.abs(
        dct(
            signal,
            norm="ortho"
        )
    )

    return energy_features(coeffs, n_bins=8)

In [ ]:
# ==========================================================
# STFT PARAMETERS
# ==========================================================

STFT_WINDOW = "hann"

STFT_NPERSEG = 1024

STFT_OVERLAP = 512

STFT_NFFT = 2048

In [ ]:
# ==========================================================
# STFT FEATURE EXTRACTION
# ==========================================================

def extract_stft_features(signal):
    """
    Extract 72 normalized STFT features.

    Procedure:

    STFT
      ↓
    Magnitude
      ↓
    Max over time
      ↓
    72 energy bins
    """

    _, _, Zxx = stft(

        signal,

        window=STFT_WINDOW,

        nperseg=STFT_NPERSEG,

        noverlap=STFT_OVERLAP,

        nfft=STFT_NFFT,

        boundary=None,

        padded=False

    )

    magnitude = np.abs(Zxx)

    spectrum = np.max(

        magnitude,

        axis=1

    )

    return energy_features(

        spectrum,

        n_bins=72

    )

In [ ]:
signal = load_dat(dataset[0]["filepath"])

fft_features = extract_fft_features(signal)

dct_features = extract_dct_features(signal)

stft_features = extract_stft_features(signal)

print("FFT :", fft_features.shape)

print("DCT :", dct_features.shape)

print("STFT :", stft_features.shape)

print()

print("FFT Energy :", fft_features.sum())

print("DCT Energy :", dct_features.sum())

print("STFT Energy :", stft_features.sum())

FFT : (8,)
DCT : (8,)
STFT : (72,)

FFT Energy : 0.9999999999999999
DCT Energy : 0.9999999999999999
STFT Energy : 1.0


In [ ]:
import time

signal = load_dat(dataset[0]["filepath"])

# FFT
start = time.time()
fft_features = extract_fft_features(signal)
print(f"FFT  : {time.time()-start:.3f} sec")

# DCT
start = time.time()
dct_features = extract_dct_features(signal)
print(f"DCT  : {time.time()-start:.3f} sec")

# STFT
start = time.time()
stft_features = extract_stft_features(signal)
print(f"STFT : {time.time()-start:.3f} sec")

FFT  : 0.001 sec
DCT  : 0.001 sec
STFT : 0.005 sec


In [ ]:
combined = np.concatenate([

    fft_features,

    dct_features,

    stft_features

])

print("Total Features :", len(combined))

Total Features : 88


In [ ]:
!pip install -q --no-deps tftb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.3 MB/s eta 0:00:00


In [ ]:
from tftb.processing import WignerVilleDistribution

print("tftb imported successfully!")

tftb imported successfully!


In [ ]:
# ==========================================================
# WAVELET PACKET TRANSFORM (254 FEATURES)
# ==========================================================

def extract_wpt_features(signal):

    wp = pywt.WaveletPacket(
        data=signal,
        wavelet='db4',
        mode='symmetric',
        maxlevel=7
    )

    features = []

    for level in range(1, 8):

        nodes = wp.get_level(level, order='natural')

        for node in nodes:

            energy = np.sum(node.data ** 2)

            features.append(energy)

    features = np.asarray(features, dtype=np.float64)

    assert len(features) == 254

    features /= np.sum(features)

    return features

In [ ]:
# ==========================================================
# WVD PARAMETERS
# ==========================================================

WVD_SIGNAL_LENGTH = 8192

In [ ]:
from tftb.processing import WignerVilleDistribution

# ==========================================================
# WVD FEATURE EXTRACTION
# ==========================================================

def extract_wvd_features(signal):

    # ------------------------------------
    # Downsample ONLY for WVD
    # ------------------------------------

    signal = resample(signal, WVD_SIGNAL_LENGTH)

    signal = signal.astype(np.float64)

    # ------------------------------------
    # Compute WVD
    # ------------------------------------

    wvd = WignerVilleDistribution(signal)

    tfr, _, _ = wvd.run()

    magnitude = np.abs(tfr)

    # ------------------------------------
    # Maximum along time axis
    # ------------------------------------

    spectrum = np.max(

        magnitude,

        axis=1

    )

    # ------------------------------------
    # 72 normalized energy bins
    # ------------------------------------

    return energy_features(

        spectrum,

        n_bins=72

    )

In [ ]:
signal = load_dat(dataset[0]["filepath"])

wpt_features = extract_wpt_features(signal)

wvd_features = extract_wvd_features(signal)

print("WPT :", wpt_features.shape)

print("WVD :", wvd_features.shape)

print()

print("WPT Energy :", np.sum(wpt_features))

print("WVD Energy :", np.sum(wvd_features))

WPT : (254,)
WVD : (72,)

WPT Energy : 1.0
WVD Energy : 1.0


In [ ]:
import time

signal = load_dat(dataset[0]["filepath"])

start = time.time()

wpt = extract_wpt_features(signal)

print(f"WPT : {time.time()-start:.2f} sec")

start = time.time()

wvd = extract_wvd_features(signal)

print(f"WVD : {time.time()-start:.2f} sec")

WPT : 0.00 sec
WVD : 2.70 sec


In [ ]:
signal = load_dat(dataset[0]["filepath"])

fft_features = extract_fft_features(signal)

dct_features = extract_dct_features(signal)

stft_features = extract_stft_features(signal)

wpt_features = extract_wpt_features(signal)

wvd_features = extract_wvd_features(signal)

all_features = np.concatenate([

    fft_features,

    dct_features,

    stft_features,

    wvd_features,

    wpt_features

])

print("FFT  :", len(fft_features))

print("DCT  :", len(dct_features))

print("STFT :", len(stft_features))

print("WVD  :", len(wvd_features))

print("WPT  :", len(wpt_features))

print()

print("TOTAL FEATURES :", len(all_features))

FFT  : 8
DCT  : 8
STFT : 72
WVD  : 72
WPT  : 254

TOTAL FEATURES : 414


In [ ]:
# ==========================================================
# COLUMN NAMES
# ==========================================================

FEATURE_COLUMNS = []

FEATURE_COLUMNS += [f"FFT_{i+1}" for i in range(8)]
FEATURE_COLUMNS += [f"DCT_{i+1}" for i in range(8)]
FEATURE_COLUMNS += [f"STFT_{i+1}" for i in range(72)]
FEATURE_COLUMNS += [f"WVD_{i+1}" for i in range(72)]
FEATURE_COLUMNS += [f"WPT_{i+1}" for i in range(254)]

print("Total feature columns:", len(FEATURE_COLUMNS))

Total feature columns: 414


In [40]:
# ==========================================================
# EXTRACT FEATURES FROM ALL FILES
# ==========================================================

results = []

checkpoint_interval = 100

for idx, item in enumerate(tqdm(dataset)):

    try:

        signal = load_dat(item["filepath"])

        fft_features = extract_fft_features(signal)

        dct_features = extract_dct_features(signal)

        stft_features = extract_stft_features(signal)

        wvd_features = extract_wvd_features(signal)

        wpt_features = extract_wpt_features(signal)

        features = np.concatenate([

            fft_features,

            dct_features,

            stft_features,

            wvd_features,

            wpt_features

        ])

        assert len(features) == 414

        row = {

            "Filename": item["filename"],

            "Class": item["class"],

            "Label": item["label"]

        }

        for name, value in zip(FEATURE_COLUMNS, features):

            row[name] = value

        results.append(row)

        # Save checkpoint every 100 files
        if (idx + 1) % checkpoint_interval == 0:

            checkpoint_df = pd.DataFrame(results)

            checkpoint_df.to_csv(

                CHECKPOINT_CSV,

                index=False

            )

            print(f"\nCheckpoint saved ({idx+1} files).")

    except Exception as e:

        print(f"\nError processing {item['filename']}")

        print(e)

  0%|          | 0/1800 [00:00<?, ?it/s]


Checkpoint saved (100 files).

Checkpoint saved (200 files).

Checkpoint saved (300 files).

Checkpoint saved (400 files).

Checkpoint saved (500 files).

Checkpoint saved (600 files).

Checkpoint saved (700 files).

Checkpoint saved (800 files).

Checkpoint saved (900 files).

Checkpoint saved (1000 files).

Checkpoint saved (1100 files).

Checkpoint saved (1200 files).

Checkpoint saved (1300 files).

Checkpoint saved (1400 files).

Checkpoint saved (1500 files).

Checkpoint saved (1600 files).

Checkpoint saved (1700 files).

Checkpoint saved (1800 files).


In [41]:
# ==========================================================
# SAVE FINAL CSV
# ==========================================================

features_df = pd.DataFrame(results)

features_df.to_csv(

    OUTPUT_CSV,

    index=False

)

print()

print("CSV saved successfully!")

print()

print("Location:")

print(OUTPUT_CSV)


CSV saved successfully!

Location:
/content/drive/MyDrive/EE656project/features.csv


In [42]:
features_df = pd.read_csv(OUTPUT_CSV)

print(features_df.shape)

features_df.head()

(1800, 417)


,Filename,Class,Label,FFT_1,FFT_2,FFT_3,FFT_4,FFT_5,FFT_6,FFT_7,...,WPT_245,WPT_246,WPT_247,WPT_248,WPT_249,WPT_250,WPT_251,WPT_252,WPT_253,WPT_254
0,preprocess_Reading1.dat,Bearing,0,0.889793,0.108674,0.001519,0.000013,2.229360e-07,5.983129e-08,3.312147e-08,...,1.702067e-06,1.472837e-06,9.786639e-08,4.873545e-08,7.657010e-08,4.821368e-08,3.336592e-07,3.188135e-07,1.016675e-07,2.244914e-07
1,preprocess_Reading10.dat,Bearing,0,0.961784,0.037361,0.000839,0.000014,7.887805e-07,4.436992e-07,4.114331e-07,...,6.424626e-07,3.876736e-07,3.492427e-08,4.013606e-08,5.761115e-08,3.072175e-08,1.518751e-07,1.646307e-07,5.444462e-08,1.209364e-07
2,preprocess_Reading100.dat,Bearing,0,0.957625,0.041133,0.001226,0.000013,9.488981e-07,6.045668e-07,6.245363e-07,...,7.616532e-07,6.270445e-07,3.367451e-08,4.153442e-08,6.179419e-08,3.832891e-08,2.064276e-07,2.059422e-07,7.244637e-08,1.188347e-07
3,preprocess_Reading101.dat,Bearing,0,0.947063,0.051486,0.001435,0.000015,3.060117e-07,7.444310e-08,6.203950e-08,...,9.227642e-07,6.280834e-07,3.924307e-08,6.267721e-08,7.199818e-08,5.239665e-08,2.998537e-07,2.042189e-07,8.828062e-08,1.498647e-07
4,preprocess_Reading102.dat,Bearing,0,0.927541,0.070705,0.001708,0.000027,5.839320e-06,4.441925e-06,4.074159e-06,...,8.952590e-07,6.314783e-07,4.812828e-08,6.192606e-08,7.068038e-08,5.250307e-08,2.834131e-07,2.978103e-07,1.126451e-07,1.761465e-07
